# EXP-00: Pipeline Preprocessing Eksperimen (C1–C4)

Notebook ini mengimplementasikan **pipeline preprocessing terpadu** dengan 4 kondisi eksperimen
untuk mengukur pengaruh setiap keputusan pembersihan data terhadap kualitas evaluasi model TransH.

## Kondisi Eksperimen

| Kondisi | Label | User ≥10 (V1) | Filter Dangling Nodes (V2) | has_tag + release_year (V3) |
|:---:|:---:|:---:|:---:|:---:|
| **C1** | `FULL_CLEAN` | ✅ | ✅ | ✅ |
| **C2** | `NO_TAG_YEAR` | ✅ | ✅ | ❌ |
| **C3** | `NO_FILTER` | ❌ (k≥3 teknis) | ❌ | ❌ |
| **C4** | `ONLY_USER_FILTER` | ✅ | ❌ | ❌ |

## Output

Untuk setiap kondisi, akan dihasilkan folder:
```
Pipeline_Experiments/<CONDITION>/
    train_triplets.tsv          ← format string
    valid_triplets.tsv
    test_triplets.tsv
    train_triplets_int.tsv      ← format integer (untuk TransH)
    valid_triplets_int.tsv
    test_triplets_int.tsv
    entity2id.txt / .json
    relation2id.txt / .json
    stats_summary.json          ← statistik graf
```

In [1]:
# ══════════════════════════════════════════════════════════════════════════
# KONFIGURASI EKSPERIMEN
# ══════════════════════════════════════════════════════════════════════════

# Pilih kondisi aktif: "C1_FULL_CLEAN" | "C2_NO_TAG_YEAR" | "C3_NO_FILTER" | "C4_ONLY_USER_FILTER"
ACTIVE_CONDITION = "C1_FULL_CLEAN"

# Set True untuk menjalankan SEMUA kondisi sekaligus
RUN_ALL_CONDITIONS = False

EXPERIMENT_CONFIG = {
    "C1_FULL_CLEAN": {
        "description": "Pipeline bersih penuh: User≥20, Filter dangling nodes, + has_tag & release_year",
        "min_user_interactions": 20,   # V1: threshold minimal interaksi user
        "filter_dangling_nodes": True, # V2: hapus aktor/direktur < 2 film
        "min_entity_freq": 2,          # V2: threshold frekuensi entitas
        "add_has_tag": True,           # V3a: relasi tag genome
        "add_release_year": True,      # V3b: relasi tahun rilis
        "technical_min_loo": 3,        # minimum teknis untuk LOO split (harus ≥3)
    },
    "C2_NO_TAG_YEAR": {
        "description": "Bersih tanpa relasi tambahan: User≥10, Filter dangling nodes, tanpa has_tag & release_year",
        "min_user_interactions": 10,
        "filter_dangling_nodes": True,
        "min_entity_freq": 2,
        "add_has_tag": False,
        "add_release_year": False,
        "technical_min_loo": 3,
    },
    "C3_NO_FILTER": {
        "description": "Tanpa pembersihan: K≥3 teknis (untuk LOO), tanpa filter dangling, tanpa relasi tambahan",
        "min_user_interactions": 3,    # Minimum teknis agar LOO valid
        "filter_dangling_nodes": False,
        "min_entity_freq": 1,
        "add_has_tag": False,
        "add_release_year": False,
        "technical_min_loo": 3,
    },
    "C4_ONLY_USER_FILTER": {
        "description": "Hanya filter user: User≥10, tanpa filter dangling, tanpa relasi tambahan",
        "min_user_interactions": 10,
        "filter_dangling_nodes": False,
        "min_entity_freq": 1,
        "add_has_tag": False,
        "add_release_year": False,
        "technical_min_loo": 3,
    },
}

print("=" * 70)
print(f"EKSPERIMEN PREPROCESSING PIPELINE")
print("=" * 70)
if RUN_ALL_CONDITIONS:
    print("Mode: JALANKAN SEMUA KONDISI")
    print(f"Kondisi yang akan dijalankan: {list(EXPERIMENT_CONFIG.keys())}")
else:
    cfg = EXPERIMENT_CONFIG[ACTIVE_CONDITION]
    print(f"Mode: Kondisi Tunggal → {ACTIVE_CONDITION}")
    print(f"Deskripsi: {cfg['description']}")

EKSPERIMEN PREPROCESSING PIPELINE
Mode: Kondisi Tunggal → C1_FULL_CLEAN
Deskripsi: Pipeline bersih penuh: User≥20, Filter dangling nodes, + has_tag & release_year


In [2]:
# ══════════════════════════════════════════════════════════════════════════
# IMPORTS & PATH KONFIGURASI
# ══════════════════════════════════════════════════════════════════════════
import pandas as pd
import numpy as np
import json
import random
from pathlib import Path
import networkx as nx

BASE_DIR = Path().resolve()
RAW_OVERLAP_DIR = BASE_DIR / "raw_overlap"
FILM_CSV        = RAW_OVERLAP_DIR / "films_metadata_clean.csv"
USER_CSV        = RAW_OVERLAP_DIR / "interactions_raw.csv"

EXP_BASE_DIR = BASE_DIR / "Pipeline_Experiments"
EXP_BASE_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print(f"Base DIR  : {BASE_DIR}")
print(f"Film CSV  : {FILM_CSV.exists()} — {FILM_CSV}")
print(f"User CSV  : {USER_CSV.exists()} — {USER_CSV}")
if not FILM_CSV.exists() or not USER_CSV.exists():
    print("❌ ERROR: File raw overlap tidak ditemukan! Jalankan EXP_PREP_Raw_Overlap.ipynb dulu.")
else:
    print("✅ raw_overlap/ tersedia. Siap eksekusi pipeline.")

Base DIR  : D:\Program-Skripsi\Preprocessing
Film CSV  : True — D:\Program-Skripsi\Preprocessing\raw_overlap\films_metadata_clean.csv
User CSV  : True — D:\Program-Skripsi\Preprocessing\raw_overlap\interactions_raw.csv
✅ raw_overlap/ tersedia. Siap eksekusi pipeline.


In [3]:
# ══════════════════════════════════════════════════════════════════════════
# FUNGSI-FUNGSI HELPER
# ══════════════════════════════════════════════════════════════════════════

def check_entity_aware(train_df, split_df, split_name):
    """Verifikasi tidak ada entitas asing di split (Entity-Aware check)."""
    train_entities = set(train_df["head"]) | set(train_df["tail"])
    split_entities = set(split_df["head"]) | set(split_df["tail"])
    unseen = split_entities - train_entities
    if unseen:
        print(f"  ⚠ PERINGATAN: {split_name} mengandung {len(unseen)} entitas asing!")
        print(f"    Contoh: {list(unseen)[:5]}")
        return False
    else:
        print(f"  ✅ {split_name}: Entity-Aware LOLOS (0 entitas asing)")
        return True


def check_connectivity(train_df):
    """Cek keterhubungan graf menggunakan NetworkX."""
    G = nx.Graph()
    for _, row in train_df.iterrows():
        G.add_edge(row["head"], row["tail"])
    
    components = list(nx.connected_components(G))
    n_comp = len(components)
    n_nodes = G.number_of_nodes()
    n_edges = G.number_of_edges()
    
    print(f"  Jumlah node     : {n_nodes:,}")
    print(f"  Jumlah edge     : {n_edges:,}")
    print(f"  Jumlah komponen : {n_comp}")
    
    if n_comp == 1:
        print(f"  ✅ KG terhubung sempurna — 1 komponen")
    else:
        sizes = sorted([len(c) for c in components], reverse=True)
        print(f"  ⚠ KG TERPUTUS — {n_comp} komponen")
        print(f"    Komponen terbesar: {sizes[0]:,} node ({sizes[0]/n_nodes*100:.1f}%)")
    
    return n_comp, n_nodes, n_edges


def convert_to_int_triplets(df, entity2id, relation2id):
    """Konversi triplet string ke triplet integer."""
    df_int = pd.DataFrame()
    df_int["head"]     = df["head"].map(entity2id)
    df_int["relation"] = df["relation"].map(relation2id)
    df_int["tail"]     = df["tail"].map(entity2id)
    return df_int

print("✅ Fungsi-fungsi helper berhasil didefinisikan.")

✅ Fungsi-fungsi helper berhasil didefinisikan.


In [4]:
# ══════════════════════════════════════════════════════════════════════════
# LOAD DATA DASAR
# ══════════════════════════════════════════════════════════════════════════
print("Memuat data dasar...")

df_film = pd.read_csv(FILM_CSV, dtype=str)
for col in df_film.columns:
    df_film[col] = df_film[col].astype(str).str.strip().replace("nan", np.nan)

df_user_raw = pd.read_csv(USER_CSV, dtype=str)
df_user_raw["user_id"] = df_user_raw["user_id"].astype(str).str.strip()
df_user_raw["show_id"] = df_user_raw["show_id"].astype(str).str.strip()

print(f"  Film metadata    : {len(df_film):,} film")
print(f"  Interaksi user   : {len(df_user_raw):,} baris ({df_user_raw['user_id'].nunique():,} user)")
print(f"✅ Data dasar berhasil dimuat.")

Memuat data dasar...
  Film metadata    : 1,301 film
  Interaksi user   : 101,633 baris (23,432 user)
✅ Data dasar berhasil dimuat.


In [5]:
# ══════════════════════════════════════════════════════════════════════════
# FUNGSI UTAMA: run_pipeline(condition_name)
# ══════════════════════════════════════════════════════════════════════════

def run_pipeline(condition_name: str):
    cfg = EXPERIMENT_CONFIG[condition_name]
    OUT_DIR = EXP_BASE_DIR / condition_name
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    
    print("\n" + "═" * 70)
    print(f"  KONDISI: {condition_name}")
    print(f"  {cfg['description']}")
    print("═" * 70)
    
    stats = {"condition": condition_name, "config": cfg.copy()}
    
    # ────────────────────────────────────────────────────────────────────
    # TAHAP 1: PREPROCESSING DATAFRAME
    # ────────────────────────────────────────────────────────────────────
    print("\n[Tahap 1] Preprocessing di tingkat DataFrame...")
    
    # 1. K-Core User & Film Filtering
    effective_min = max(cfg['min_user_interactions'], cfg['technical_min_loo'])
    interactions = df_user_raw.copy()
    iteration = 1
    while True:
        start_len = len(interactions)
        user_counts = interactions['user_id'].value_counts()
        valid_users = user_counts[user_counts >= effective_min].index
        interactions = interactions[interactions['user_id'].isin(valid_users)]
        
        movie_counts = interactions['show_id'].value_counts()
        valid_movies = movie_counts[movie_counts >= 1].index
        interactions = interactions[interactions['show_id'].isin(valid_movies)]
        
        end_len = len(interactions)
        if start_len == end_len:
            break
        iteration += 1
        
    n_users = interactions['user_id'].nunique()
    n_films = interactions['show_id'].nunique()
    n_interactions = len(interactions)
    stats["n_users"] = n_users
    stats["n_films"] = n_films
    stats["n_interactions"] = n_interactions
    stats["avg_interactions_per_user"] = round(n_interactions / n_users, 2)
    print(f"  K-Core Filter: {n_interactions:,} interaksi | {n_users:,} user | {n_films:,} film")
    
    # ────────────────────────────────────────────────────────────────────
    # TAHAP 2: PEMBAGIAN DATASET (SPLITTING DI TINGKAT DATAFRAME)
    # ────────────────────────────────────────────────────────────────────
    print("\n[Tahap 2] Dataset Splitting (LOO Split) di tingkat DataFrame...")
    
    df_user_triplets = interactions.copy()
    df_user_triplets = df_user_triplets.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
    
    test_list, valid_list, train_list = [], [], []
    for user, group in df_user_triplets.groupby('user_id'):
        test_list.append(group.iloc[0:1])
        valid_list.append(group.iloc[1:2])
        train_list.append(group.iloc[2:])
        
    df_train_interact_df = pd.concat(train_list, ignore_index=True)
    df_valid_df = pd.concat(valid_list, ignore_index=True)
    df_test_df = pd.concat(test_list, ignore_index=True)
    
    # Konversi ke format triplet
    df_train_interact = pd.DataFrame({
        'head': 'U_' + df_train_interact_df['user_id'],
        'relation': 'liked',
        'tail': df_train_interact_df['show_id']
    })
    df_valid = pd.DataFrame({
        'head': 'U_' + df_valid_df['user_id'],
        'relation': 'liked',
        'tail': df_valid_df['show_id']
    })
    df_test = pd.DataFrame({
        'head': 'U_' + df_test_df['user_id'],
        'relation': 'liked',
        'tail': df_test_df['show_id']
    })
    
    # ────────────────────────────────────────────────────────────────────
    # TAHAP 3: TRIPLET BUILDING & METADATA PREPROCESSING
    # ────────────────────────────────────────────────────────────────────
    print("\n[Tahap 3] Membangun triplet metadata film...")
    
    valid_show_ids = set(interactions['show_id'])
    df_film_filtered = df_film[df_film['show_id'].isin(valid_show_ids)].copy()
    
    # 3a. Triplet Genre
    if 'listed_in' in df_film_filtered.columns:
        df_genre_exp = df_film_filtered[['show_id', 'listed_in']].dropna()
        df_genre_exp['listed_in'] = df_genre_exp['listed_in'].apply(lambda x: [g.strip() for g in str(x).split(',') if g.strip()])
        df_genre_exp = df_genre_exp.explode('listed_in')
        df_genre_triplets = pd.DataFrame({
            'head': df_genre_exp['show_id'],
            'relation': 'has_genre',
            'tail': df_genre_exp['listed_in']
        })
    else:
        df_genre_triplets = pd.DataFrame(columns=['head', 'relation', 'tail'])
        
    # 3b. Triplet Aktor (dengan filter dangling jika aktif)
    if 'cast' in df_film_filtered.columns:
        df_cast_exp = df_film_filtered[['show_id', 'cast']].dropna()
        df_cast_exp['cast'] = df_cast_exp['cast'].apply(lambda x: [c.strip() for c in str(x).split(',') if c.strip()])
        df_cast_exp = df_cast_exp.explode('cast')
        if cfg['filter_dangling_nodes']:
            cast_counts = df_cast_exp['cast'].value_counts()
            valid_cast = cast_counts[cast_counts >= cfg['min_entity_freq']].index
            df_cast_exp = df_cast_exp[df_cast_exp['cast'].isin(valid_cast)]
        df_actor_triplets = pd.DataFrame({
            'head': df_cast_exp['show_id'],
            'relation': 'has_actor',
            'tail': df_cast_exp['cast']
        })
    else:
        df_actor_triplets = pd.DataFrame(columns=['head', 'relation', 'tail'])
        
    # 3c. Triplet Sutradara (dengan filter dangling jika aktif)
    if 'director' in df_film_filtered.columns:
        df_dir_exp = df_film_filtered[['show_id', 'director']].dropna()
        df_dir_exp['director'] = df_dir_exp['director'].apply(lambda x: [d.strip() for d in str(x).split(',') if d.strip()])
        df_dir_exp = df_dir_exp.explode('director')
        if cfg['filter_dangling_nodes']:
            dir_counts = df_dir_exp['director'].value_counts()
            valid_dir = dir_counts[dir_counts >= cfg['min_entity_freq']].index
            df_dir_exp = df_dir_exp[df_dir_exp['director'].isin(valid_dir)]
        df_director_triplets = pd.DataFrame({
            'head': df_dir_exp['show_id'],
            'relation': 'has_director',
            'tail': df_dir_exp['director']
        })
    else:
        df_director_triplets = pd.DataFrame(columns=['head', 'relation', 'tail'])
        
    # 3d. Triplet Negara
    if 'country' in df_film_filtered.columns:
        df_country_exp = df_film_filtered[['show_id', 'country']].dropna()
        df_country_exp['country'] = df_country_exp['country'].apply(lambda x: [c.strip() for c in str(x).split(',') if c.strip()])
        df_country_exp = df_country_exp.explode('country')
        df_country_triplets = pd.DataFrame({
            'head': df_country_exp['show_id'],
            'relation': 'produced_in',
            'tail': df_country_exp['country']
        })
    else:
        df_country_triplets = pd.DataFrame(columns=['head', 'relation', 'tail'])
        
    # 3e. Triplet Rating
    if 'rating' in df_film_filtered.columns:
        df_rating_exp = df_film_filtered[['show_id', 'rating']].dropna()
        df_rating_triplets = pd.DataFrame({
            'head': df_rating_exp['show_id'],
            'relation': 'has_rating',
            'tail': df_rating_exp['rating'].str.strip()
        })
    else:
        df_rating_triplets = pd.DataFrame(columns=['head', 'relation', 'tail'])
        
    # 3f. V3a: Triplet has_tag (dengan filter dangling jika aktif)
    if cfg['add_has_tag'] and 'tags' in df_film_filtered.columns:
        df_tags_exp = df_film_filtered[['show_id', 'tags']].dropna()
        df_tags_exp['tags'] = df_tags_exp['tags'].apply(lambda x: [t.strip() for t in str(x).split(',') if t.strip()])
        df_tags_exp = df_tags_exp.explode('tags')
        if cfg['filter_dangling_nodes']:
            tag_counts = df_tags_exp['tags'].value_counts()
            valid_tags = tag_counts[tag_counts >= cfg['min_entity_freq']].index
            df_tags_exp = df_tags_exp[df_tags_exp['tags'].isin(valid_tags)]
        df_tag_triplets = pd.DataFrame({
            'head': df_tags_exp['show_id'],
            'relation': 'has_tag',
            'tail': df_tags_exp['tags']
        })
        print(f"  [V3a] has_tag ditambahkan: {len(df_tag_triplets):,} triplet")
    else:
        df_tag_triplets = pd.DataFrame(columns=['head', 'relation', 'tail'])
        print(f"  [V3a] has_tag: DINONAKTIFKAN")
        
    # 3g. V3b: Triplet release_year
    if cfg['add_release_year'] and 'release_year' in df_film_filtered.columns:
        df_year_exp = df_film_filtered[['show_id', 'release_year']].dropna()
        df_year_exp['release_year'] = df_year_exp['release_year'].apply(lambda x: str(int(float(str(x).strip()))))
        df_year_triplets = pd.DataFrame({
            'head': df_year_exp['show_id'],
            'relation': 'release_year',
            'tail': df_year_exp['release_year']
        })
        print(f"  [V3b] release_year ditambahkan: {len(df_year_triplets):,} triplet")
    else:
        df_year_triplets = pd.DataFrame(columns=['head', 'relation', 'tail'])
        print(f"  [V3b] release_year: DINONAKTIFKAN")
        
    # Gabungkan semua metadata
    df_all_metadata = pd.concat([
        df_genre_triplets,
        df_actor_triplets,
        df_director_triplets,
        df_country_triplets,
        df_rating_triplets,
        df_tag_triplets,
        df_year_triplets
    ], ignore_index=True).drop_duplicates().reset_index(drop=True)
    
    print(f"  Total triplet metadata: {len(df_all_metadata):,}")
    stats["n_metadata_triplets"] = len(df_all_metadata)
    for rel, cnt in df_all_metadata['relation'].value_counts().items():
        stats[f"triplets_{rel}"] = int(cnt)
        
    # ────────────────────────────────────────────────────────────────────
    # TAHAP 4: KEPATUHAN ENTITY-AWARE & KONEKTIVITAS GRAF
    # ────────────────────────────────────────────────────────────────────
    print("\n[Tahap 4] Penggabungan Data & Verifikasi Graf...")
    
    df_train = pd.concat([df_all_metadata, df_train_interact], ignore_index=True).drop_duplicates().reset_index(drop=True)
    print(f"  Train total   : {len(df_train):,} triplet")
    
    ea_valid = check_entity_aware(df_train, df_valid, "Valid")
    ea_test  = check_entity_aware(df_train, df_test,  "Test")
    stats["entity_aware_valid"] = ea_valid
    stats["entity_aware_test"]  = ea_test
    
    n_comp, n_nodes, n_edges = check_connectivity(df_train)
    stats["n_components"]  = n_comp
    stats["n_graph_nodes"] = n_nodes
    stats["n_graph_edges"] = n_edges
    
    # ────────────────────────────────────────────────────────────────────
    # TAHAP 5: KONVERSI INTEGER & PENYIMPANAN
    # ────────────────────────────────────────────────────────────────────
    print("\n[Tahap 5] Menyimpan kamus ID dan triplet...")
    
    df_train = df_train.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
    df_valid = df_valid.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
    df_test  = df_test.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
    
    all_triplets = pd.concat([df_train, df_valid, df_test], ignore_index=True)
    entities  = sorted(list(set(all_triplets["head"]) | set(all_triplets["tail"])))
    relations = sorted(list(all_triplets["relation"].unique()))
    
    entity2id  = {e: i for i, e in enumerate(entities)}
    relation2id = {r: i for i, r in enumerate(relations)}
    
    stats["n_entities"]  = len(entities)
    stats["n_relations"] = len(relations)
    stats["n_train_triplets"] = len(df_train)
    stats["n_valid_triplets"] = len(df_valid)
    stats["n_test_triplets"]  = len(df_test)
    stats["relations_list"] = relations
    
    # String tsv
    df_train.to_csv(OUT_DIR / "train_triplets.tsv", sep="\t", header=False, index=False)
    df_valid.to_csv(OUT_DIR / "valid_triplets.tsv", sep="\t", header=False, index=False)
    df_test.to_csv(OUT_DIR  / "test_triplets.tsv",  sep="\t", header=False, index=False)
    
    # Integer tsv
    convert_to_int_triplets(df_train, entity2id, relation2id).to_csv(OUT_DIR / "train_triplets_int.tsv", sep="\t", header=False, index=False)
    convert_to_int_triplets(df_valid, entity2id, relation2id).to_csv(OUT_DIR / "valid_triplets_int.tsv", sep="\t", header=False, index=False)
    convert_to_int_triplets(df_test, entity2id, relation2id).to_csv(OUT_DIR / "test_triplets_int.tsv", sep="\t", header=False, index=False)
    
    # Kamus
    with open(OUT_DIR / "entity2id.txt", "w", encoding="utf-8") as f:
        f.write(f"{len(entity2id)}\n")
        for ent, idx in entity2id.items():
            f.write(f"{ent}\t{idx}\n")
    with open(OUT_DIR / "relation2id.txt", "w", encoding="utf-8") as f:
        f.write(f"{len(relation2id)}\n")
        for rel, idx in relation2id.items():
            f.write(f"{rel}\t{idx}\n")
            
    with open(OUT_DIR / "entity2id.json", "w", encoding="utf-8") as f:
        json.dump(entity2id, f, indent=2)
    with open(OUT_DIR / "relation2id.json", "w", encoding="utf-8") as f:
        json.dump(relation2id, f, indent=2)
        
    # Summary
    with open(OUT_DIR / "stats_summary.json", "w", encoding="utf-8") as f:
        json.dump(stats, f, indent=2)
        
    print(f"  ✅ Semua file berhasil disimpan ke: {OUT_DIR}")
    return stats


In [6]:
# ══════════════════════════════════════════════════════════════════════════
# EKSEKUSI PIPELINE
# ══════════════════════════════════════════════════════════════════════════

if RUN_ALL_CONDITIONS:
    print("🚀 Menjalankan SEMUA 4 kondisi eksperimen...")
    all_stats = {}
    for cond_name in EXPERIMENT_CONFIG.keys():
        all_stats[cond_name] = run_pipeline(cond_name)
else:
    print(f"🚀 Menjalankan kondisi tunggal: {ACTIVE_CONDITION}...")
    run_pipeline(ACTIVE_CONDITION)
    
print("\n🎉 PIPELINE EKSPERIMEN SELESAI EKSEKUSI!")

🚀 Menjalankan kondisi tunggal: C1_FULL_CLEAN...

══════════════════════════════════════════════════════════════════════
  KONDISI: C1_FULL_CLEAN
  Pipeline bersih penuh: User≥20, Filter dangling nodes, + has_tag & release_year
══════════════════════════════════════════════════════════════════════

[Tahap 1] Preprocessing di tingkat DataFrame...
  K-Core Filter: 26,152 interaksi | 812 user | 1,195 film

[Tahap 2] Dataset Splitting (LOO Split) di tingkat DataFrame...

[Tahap 3] Membangun triplet metadata film...
  [V3a] has_tag ditambahkan: 4,783 triplet
  [V3b] release_year ditambahkan: 1,195 triplet
  Total triplet metadata: 14,084

[Tahap 4] Penggabungan Data & Verifikasi Graf...
  Train total   : 38,612 triplet
  ✅ Valid: Entity-Aware LOLOS (0 entitas asing)
  ✅ Test: Entity-Aware LOLOS (0 entitas asing)
  Jumlah node     : 3,841
  Jumlah edge     : 38,599
  Jumlah komponen : 1
  ✅ KG terhubung sempurna — 1 komponen

[Tahap 5] Menyimpan kamus ID dan triplet...
  ✅ Semua file berhasil